# TP – Aplicaciones con LLMs · Opción A: RAG

**Sistema de preguntas y respuestas sobre los apuntes del Trayecto Matemática (Secretaría de Admisión, ITBA)**

Este notebook implementa de punta a punta un sistema RAG (*Retrieval-Augmented Generation*) local:

| Etapa | Herramienta |
|---|---|
| Extracción de texto de PDFs | PyMuPDF |
| Embeddings | `nomic-embed-text` vía Ollama |
| Base vectorial | ChromaDB (persistente) |
| Generación | `gemma3` vía Ollama |

**Contenido**

0. Configuración y chequeo del entorno
1. Corpus
2. Extracción de texto y corrección de acentos
3. Chunking
4. Embeddings e indexación
5. Recuperación (top-k)
6. Generación con y sin contexto
7. Evaluación CON vs. SIN RAG
8. Experimentos: sensibilidad a *k* y prefijos de `nomic-embed-text`
9. Conclusiones

> Requisitos previos: Ollama corriendo localmente con `ollama pull gemma3` y `ollama pull nomic-embed-text`, y los PDFs del curso en la carpeta `docs/`.

## 0. Configuración y chequeo del entorno

Todos los parámetros del sistema están en una sola celda para que sea fácil experimentar.

- `CHUNK_SIZE` y `CHUNK_OVERLAP` están en **caracteres**. Se eligieron chunks grandes (1500 con 300 de solapamiento) porque los apuntes tienen "cajas" de definición y ejemplos con fórmulas, y con chunks chicos el enunciado de un ejemplo quedaba separado de su resolución.
- `TEMPERATURA = 0` hace que las respuestas sean (casi) deterministas, condición necesaria para comparar CON vs. SIN RAG de forma justa.
- `EXCLUIR_PDFS` lista fragmentos de nombre de archivo que no se indexan (ver sección 1).

In [2]:
# Si hace falta instalar dependencias, descomentar:
 #%pip install -r requirements.txt

IndentationError: unexpected indent (1446491744.py, line 2)

In [3]:
from __future__ import annotations

import json
import time
import unicodedata
from pathlib import Path

import chromadb
import ollama
import pandas as pd
import pymupdf as fitz

try:
    from IPython.display import Markdown, display
except ImportError:  # permite correr el código fuera de Jupyter
    Markdown = str
    display = print

# --- Rutas (relativas a la carpeta del notebook) ---
BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "docs"
PERSIST_DIR = BASE_DIR / "chroma_tp"
RESULTADOS_DIR = BASE_DIR / "resultados"
RESULTADOS_DIR.mkdir(exist_ok=True)

# --- Modelos ---
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "gemma3"
TEMPERATURA = 0

# --- Parámetros del pipeline ---
COLLECTION_NAME = "pdf_docs_tp"
CHUNK_SIZE = 1500      # caracteres por chunk
CHUNK_OVERLAP = 300    # caracteres compartidos entre chunks consecutivos
TOP_K = 10             # chunks recuperados por consulta
REINDEXAR = False      # True: borra la colección y vuelve a indexar todo

# Documentos que se excluyen del índice (coincidencia parcial, sin importar
# mayúsculas ni acentos). "Matrices y Sistemas" es la unión de los apuntes
# "Matrices 1C 2026" y "Sistemas 1C 2026": indexarlo duplicaría contenido.
EXCLUIR_PDFS = ["matrices y sistemas"]

pd.set_option("display.max_colwidth", 120)

In [4]:
# Chequeo de que Ollama está corriendo y tiene los modelos necesarios
try:
    modelos = ollama.list()["models"]
    nombres = [
        (m.get("model") or m.get("name")) if isinstance(m, dict) else getattr(m, "model", None)
        for m in modelos
    ]
except Exception as e:  # noqa: BLE001
    raise RuntimeError(
        "No se pudo conectar con Ollama. ¿Está corriendo? (ollama serve)"
    ) from e

for modelo in (EMBED_MODEL, CHAT_MODEL):
    ok = any(n and n.split(":")[0] == modelo for n in nombres)
    print(f"{'OK ' if ok else 'FALTA'} {modelo}" + ("" if ok else f"  ->  ollama pull {modelo}"))

OK  nomic-embed-text
OK  gemma3


## 1. Corpus

El corpus son los apuntes teóricos del Trayecto Matemática de la Secretaría de Admisión del ITBA, generados con LaTeX: Lógica, Conjuntos, Números reales, Demostraciones, Números complejos, Expresiones algebraicas, Ecuaciones e inecuaciones, Plano cartesiano, Funciones, Trigonometría, Matrices y Sistemas de ecuaciones.

**Decisión de curación:** el archivo *T12. Matrices y Sistemas* contiene, casi sin cambios, el texto de *T12. Matrices 1C 2026* y de *T13. Sistemas 1C 2026* (más del 80 % de sus oraciones aparecen en los otros dos). Si se indexan los tres, una consulta sobre sistemas de ecuaciones recupera dos copias de cada fragmento y el top-k pierde diversidad. Por eso se excluye.

In [5]:
def normalizar(texto: str) -> str:
    # Minúsculas y sin acentos: para comparar nombres de archivo y buscar evidencia.
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def _solo_palabras(texto: str) -> str:
    # "T12._Matrices_y_Sistemas.pdf" -> "t12 matrices y sistemas pdf"
    return " ".join("".join(c if c.isalnum() else " " for c in normalizar(texto)).split())


def esta_excluido(nombre: str) -> bool:
    return any(_solo_palabras(patron) in _solo_palabras(nombre) for patron in EXCLUIR_PDFS)


pdfs = sorted(DOCS_DIR.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No hay PDFs en {DOCS_DIR}")

filas = []
for ruta in pdfs:
    with fitz.open(ruta) as doc:
        filas.append({"archivo": ruta.name, "páginas": doc.page_count,
                      "indexar": not esta_excluido(ruta.name)})
corpus = pd.DataFrame(filas)
print(f"{corpus['indexar'].sum()} documentos a indexar "
      f"({len(corpus)} en la carpeta, {(~corpus['indexar']).sum()} excluidos)")
corpus

12 documentos a indexar (13 en la carpeta, 1 excluidos)


,archivo,páginas,indexar
0,T01. Lógica Proposicional.pdf,36,True
1,T02. Conjuntos.pdf,35,True
2,T03. Números reales.pdf,50,True
3,T04. Demostraciones.pdf,20,True
4,T05. Números Complejos.pdf,20,True
5,T06. Expresiones Algebraicas Polinómicas.pdf,30,True
6,T07. Exp Alg no polinómicas. Ecuaciones e Inecuaciones.pdf,30,True
7,T08. Plano cartesiano.pdf,57,True
8,T09. Funciones.pdf,74,True
9,T11. Trigonometría.pdf,60,True


## 2. Extracción de texto y corrección de acentos

Dos problemas aparecieron con estos PDFs y condicionaron el diseño:

1. **Palabras pegadas.** Con `pypdf`, la extracción unía palabras sin espacio (por ejemplo, *denominatangente*). Eso rompe justamente los términos clave que aparecen en las preguntas. **PyMuPDF** respeta los espacios.
2. **Acentos sueltos.** La fuente de LaTeX codifica "ó" como dos glifos: el acento suelto y la vocal (`´o`). El texto queda como *Definici´on*, que el modelo de embeddings no reconoce como *Definición*. Se corrige con un mapeo de reemplazos antes de indexar.

Además, se normalizan espacios y saltos de línea.

In [6]:
# El glifo de acento aparece SUELTO antes de la vocal, en vez de combinado.
ACCENT_FIX = {
    "´a": "á", "´e": "é", "´ı": "í", "´o": "ó", "´u": "ú",
    "´A": "Á", "´E": "É", "´I": "Í", "´O": "Ó", "´U": "Ú",
    "¨u": "ü", "˜n": "ñ",
}


def fix_accents(texto: str) -> str:
    for roto, bien in ACCENT_FIX.items():
        texto = texto.replace(roto, bien)
    return texto


# Antes / después sobre una página real del corpus
with fitz.open(pdfs[0]) as doc:
    crudo = " ".join(doc[min(2, doc.page_count - 1)].get_text().split())[:300]
print("ANTES:  ", crudo)
print()
print("DESPUÉS:", fix_accents(crudo))

ANTES:   L´ogica Proposicional 1. Introducci´on Como todo juego, la Matem´atica tiene sus reglas. Se construye a partir de elementos b´asicos (por ejemplo: el punto, el n´umero, etc.) y de una m´ınima cantidad de postulados aceptados como verdaderos sin necesidad de demostraci´on (por ejemplo: “Por dos punto

DESPUÉS: Lógica Proposicional 1. Introducción Como todo juego, la Matemática tiene sus reglas. Se construye a partir de elementos básicos (por ejemplo: el punto, el número, etc.) y de una mínima cantidad de postulados aceptados como verdaderos sin necesidad de demostración (por ejemplo: “Por dos punto


## 3. Chunking

**Estrategia:** ventana deslizante de tamaño fijo (`CHUNK_SIZE` caracteres) con solapamiento (`CHUNK_OVERLAP`), aplicada sobre el **texto completo** de cada PDF y no página por página.

¿Por qué sobre el documento completo? Muchos ejemplos de los apuntes empiezan en una página y terminan en la siguiente (por ejemplo, el de la recta tangente a una circunferencia en *Plano cartesiano*). Chunkear por página dejaba el enunciado en un chunk y la resolución en otro.

Para poder citar la página igual, se guarda un mapa *offset de carácter → número de página*, y cada chunk se etiqueta con la página donde **empieza**.

In [7]:
def chunk_text_with_offsets(texto: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    # Devuelve [(offset_inicial, texto_del_chunk), ...]
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP debe ser menor que CHUNK_SIZE")
    chunks, inicio = [], 0
    while inicio < len(texto):
        chunks.append((inicio, texto[inicio:inicio + size]))
        inicio += size - overlap
    return chunks


def extraer_texto_completo(ruta: Path):
    # Texto de todo el PDF + mapa [(offset_inicial, n_pagina), ...]
    partes, mapa_paginas, offset = [], [], 0
    with fitz.open(ruta) as doc:
        for n_pagina, page in enumerate(doc, start=1):
            texto = " ".join(fix_accents(page.get_text() or "").split())
            if not texto:
                continue
            mapa_paginas.append((offset, n_pagina))
            partes.append(texto)
            offset += len(texto) + 1  # +1 por el espacio que une las páginas
    return " ".join(partes), mapa_paginas


def pagina_para_offset(offset: int, mapa_paginas) -> int:
    pagina = mapa_paginas[0][1] if mapa_paginas else 1
    for offset_inicio, n_pagina in mapa_paginas:
        if offset_inicio > offset:
            break
        pagina = n_pagina
    return pagina


def extraer_chunks_pdf(ruta: Path) -> list[dict]:
    texto, mapa = extraer_texto_completo(ruta)
    return [
        {"id": f"{ruta.name}::c{i}", "text": ch,
         "metadata": {"source": ruta.name, "page": pagina_para_offset(off, mapa)}}
        for i, (off, ch) in enumerate(chunk_text_with_offsets(texto))
    ]


chunks_por_doc = {r.name: extraer_chunks_pdf(r) for r in pdfs if not esta_excluido(r.name)}
resumen = pd.DataFrame(
    [{"archivo": n, "chunks": len(c), "caracteres indexados": sum(len(x["text"]) for x in c)}
     for n, c in chunks_por_doc.items()]
)
print("Total de chunks:", resumen["chunks"].sum())
resumen

Total de chunks: 394


,archivo,chunks,caracteres indexados
0,T01. Lógica Proposicional.pdf,36,53676
1,T02. Conjuntos.pdf,29,42303
2,T03. Números reales.pdf,51,75192
3,T04. Demostraciones.pdf,20,28947
4,T05. Números Complejos.pdf,19,26726
5,T06. Expresiones Algebraicas Polinómicas.pdf,33,48146
6,T07. Exp Alg no polinómicas. Ecuaciones e Inecuaciones.pdf,26,37843
7,T08. Plano cartesiano.pdf,47,69778
8,T09. Funciones.pdf,63,94085
9,T11. Trigonometría.pdf,52,77389


In [8]:
# Un chunk de ejemplo, con su metadata
ejemplo = chunks_por_doc[next(iter(chunks_por_doc))][3]
print(ejemplo["metadata"])
print(ejemplo["text"][:700], "...")

{'source': 'T01. Lógica Proposicional.pdf', 'page': 3}
lógica es impersonal, desconoce ambigüedades y supuestos. Debe poder analizarse del mismo modo bajo cualquier contexto y por cualquier persona. Para evitar distintas interpre- taciones, generó sus propios códigos y sus propios símbolos. 3 Lógica Proposicional 2.1. Proposiciones y su valor de verdad Definición Se denominan proposiciones a los enunciados de los que se puede asegurar si son verdaderos o falsos. Estos son los tipos de enunciados que reconoce la lógica. Ejemplo 2.a) La capital de la provincia de La Pampa es Santa Rosa. Esta es una proposición verdadera. ¿Llegó Juan? Esta no es una proposición pues, haya o no llegado Juan, de una pregunta nunca podemos decir que sea verdadera o fa ...


## 4. Embeddings e indexación

Cada chunk se convierte en un vector con `nomic-embed-text` y se guarda en una colección **persistente** de ChromaDB. Así, la indexación (la etapa más lenta) se hace una sola vez; las ejecuciones siguientes reutilizan la base.

- La colección usa distancia **coseno** (`hnsw:space = cosine`), la medida habitual para embeddings de texto.
- Se usa la misma función de embedding para documentos y consultas; si no, los vectores no son comparables.
- `nomic-embed-text` fue entrenado con prefijos de tarea (`search_document:` / `search_query:`). El pipeline base **no** los usa, y en la sección 8 se mide si agregarlos mejora la recuperación.

In [9]:
def embed_textos(textos: list[str], prefijo: str = "") -> list[list[float]]:
    # Embeddings en lote; si la versión de la librería no tiene `embed`, uno por uno.
    textos = [prefijo + t for t in textos]
    if hasattr(ollama, "embed"):
        vectores = []
        for i in range(0, len(textos), 32):
            vectores.extend(ollama.embed(model=EMBED_MODEL, input=textos[i:i + 32])["embeddings"])
        return vectores
    return [ollama.embeddings(model=EMBED_MODEL, prompt=t)["embedding"] for t in textos]


def get_collection(nombre: str, reindexar: bool = False):
    client = chromadb.PersistentClient(path=str(PERSIST_DIR))
    if reindexar:
        try:
            client.delete_collection(nombre)
        except Exception:  # noqa: BLE001 - no existía
            pass
    return client.get_or_create_collection(nombre, metadata={"hnsw:space": "cosine"})


def indexar(collection, chunks_por_doc: dict, prefijo_doc: str = "") -> None:
    nuevos = 0
    for nombre, items in chunks_por_doc.items():
        if collection.get(where={"source": nombre}, limit=1)["ids"]:
            print(f"  = {nombre} (ya indexado)")
            continue
        t0 = time.time()
        collection.add(
            ids=[it["id"] for it in items],
            embeddings=embed_textos([it["text"] for it in items], prefijo_doc),
            documents=[it["text"] for it in items],
            metadatas=[it["metadata"] for it in items],
        )
        nuevos += len(items)
        print(f"  + {nombre}: {len(items)} chunks ({time.time() - t0:.1f}s)")
    print(f"\nChunks nuevos: {nuevos}. Total en la colección: {collection.count()}")


collection = get_collection(COLLECTION_NAME, reindexar=REINDEXAR)
indexar(collection, chunks_por_doc)

  + T01. Lógica Proposicional.pdf: 36 chunks (3.3s)
  + T02. Conjuntos.pdf: 29 chunks (2.3s)
  + T03. Números reales.pdf: 51 chunks (4.2s)
  + T04. Demostraciones.pdf: 20 chunks (1.6s)
  + T05. Números Complejos.pdf: 19 chunks (1.4s)
  + T06. Expresiones Algebraicas Polinómicas.pdf: 33 chunks (2.5s)
  + T07. Exp Alg no polinómicas. Ecuaciones e Inecuaciones.pdf: 26 chunks (2.1s)
  + T08. Plano cartesiano.pdf: 47 chunks (3.8s)
  + T09. Funciones.pdf: 63 chunks (5.0s)
  + T11. Trigonometría.pdf: 52 chunks (4.3s)
  + T12. Matrices (1C 2026).pdf: 5 chunks (0.4s)
  + T13. Sistemas (1C 2026).pdf: 13 chunks (1.1s)

Chunks nuevos: 394. Total en la colección: 394


## 5. Recuperación (top-k)

La consulta se convierte en vector con el mismo modelo y Chroma devuelve los `TOP_K` chunks más cercanos. Se muestra la distancia coseno (menor = más parecido) para poder inspeccionar qué trajo el retriever **antes** de mirar la respuesta del modelo.

In [10]:
def recuperar(collection, pregunta: str, k: int = TOP_K, prefijo_query: str = "") -> list[dict]:
    q = embed_textos([pregunta], prefijo_query)[0]
    res = collection.query(query_embeddings=[q], n_results=k)
    return [
        {"text": d, "meta": m, "distancia": dist}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


def tabla_contexto(contexto: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([
        {"#": i + 1, "fuente": c["meta"]["source"], "pág.": c["meta"]["page"],
         "distancia": round(c["distancia"], 3), "inicio del chunk": c["text"][:90] + "..."}
        for i, c in enumerate(contexto)
    ])


pregunta_demo = "¿Qué es una recta tangente a una circunferencia?"
contexto_demo = recuperar(collection, pregunta_demo)
tabla_contexto(contexto_demo)

,#,fuente,pág.,distancia,inicio del chunk
0,1,T08. Plano cartesiano.pdf,45,0.208,t. A t B Ejemplo 7.a) Hallar una recta tangente a la circunferencia (x −2)2 + (y −2)2 = 9...
1,2,T08. Plano cartesiano.pdf,49,0.230,B C x y Mediatriz AB Mediatriz BC 49 Plano Cartesiano Posiciones relativas entre dos circ...
2,3,T08. Plano cartesiano.pdf,19,0.232,"a circunferencia en forma canónica, que es aquella en la que el centro está a la vista. La..."
3,4,T08. Plano cartesiano.pdf,47,0.241,as rectas serán secantes si sus pendientes son menores que √ 3 o mayores que √ 3. Si ∆= 0:...
4,5,T08. Plano cartesiano.pdf,43,0.251,"nante menor que cero, no existen soluciones reales. De modo que no hay intersección entre ..."
5,6,T08. Plano cartesiano.pdf,46,0.285,"tersección. Observemos que las rectas buscadas, por contener al origen, tienen como ecuaci..."
6,7,T08. Plano cartesiano.pdf,48,0.312,"AB: El punto medio del segmento AB es ( 1 2, 11 2 ). La pendiente de la recta que contien..."
7,8,T08. Plano cartesiano.pdf,32,0.318,"ella, todas con pen- diente m = −1 2. Ejemplo 5.b) Encontremos dos rectas (distintas) per..."
8,9,T11. Trigonometría.pdf,17,0.322,gitudes del arco que determina un cierto ángulo sobre la circunferencia trigonométrica. La...
9,10,T09. Funciones.pdf,72,0.323,"Recordemos que, en la práctica de plano cartesiano, definimos a la circunferencia co- mo ..."


## 6. Generación con y sin contexto

Se comparan dos formas de responder **con el mismo modelo y la misma temperatura**:

- **SIN RAG:** la pregunta va directo a `gemma3`, que solo cuenta con su conocimiento paramétrico.
- **CON RAG:** la pregunta va acompañada de los chunks recuperados. El prompt indica responder **solo** con ese contexto, decir explícitamente si no alcanza y citar archivo y página.

Esa última instrucción es la que permite evaluar si el sistema *se abstiene* ante preguntas que el corpus no cubre en lugar de inventar.

In [11]:
OPCIONES_LLM = {"temperature": TEMPERATURA}


def responder_sin_rag(pregunta: str) -> str:
    resp = ollama.chat(model=CHAT_MODEL, options=OPCIONES_LLM,
                       messages=[{"role": "user", "content": pregunta}])
    return resp["message"]["content"]


def responder_con_rag(pregunta: str, contexto: list[dict]) -> str:
    bloques = [f"[{c['meta']['source']} (pág. {c['meta']['page']})]\n{c['text']}" for c in contexto]
    prompt = (
        "Respondé la pregunta usando SOLO la información del contexto extraído de los "
        "apuntes. Si el contexto no alcanza para responder, decilo explícitamente. "
        "Citá las fuentes entre corchetes.\n\n"
        "Contexto:\n" + "\n\n".join(bloques) +
        f"\n\nPregunta: {pregunta}\n\nRespuesta:"
    )
    resp = ollama.chat(model=CHAT_MODEL, options=OPCIONES_LLM,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"]


display(Markdown("### SIN RAG\n" + responder_sin_rag(pregunta_demo)))
display(Markdown("### CON RAG\n" + responder_con_rag(pregunta_demo, contexto_demo)))

### SIN RAG
Una recta tangente a una circunferencia es una recta que **toca la circunferencia en un solo punto** y que es **tangente** a la circunferencia en ese punto.  Esto significa que la recta forma un ángulo de 90 grados con el radio que se extiende desde el centro de la circunferencia hasta el punto de tangencia.

Aquí te explico los puntos clave:

* **Centro de la Circunferencia:** Es el punto alrededor del cual la circunferencia está centrada.
* **Radio:** Es la línea recta que va desde el centro de la circunferencia hasta cualquier punto en la circunferencia.
* **Punto de Tangencia:** Es el punto donde la recta tangente toca la circunferencia.
* **Ángulo de 90 Grados:**  El ángulo formado entre la recta tangente y el radio en el punto de tangencia es siempre un ángulo recto (90 grados).

**Imagina esto:**

Piensa en una circunferencia dibujada alrededor de un círculo.  Si dibujas una línea recta que toca la circunferencia en un solo punto, y esa línea forma un ángulo de 90 grados con el radio que va desde el centro hasta ese punto, entonces esa línea es una recta tangente.

**En términos matemáticos:**

Si el centro de la circunferencia es (h, k) y la ecuación de la circunferencia es (x - h)² + (y - k)² = r², donde 'r' es el radio, entonces la ecuación de la recta tangente en un punto (x₁, y₁) en la circunferencia se puede encontrar usando la siguiente fórmula:

(x - h)(x₁ - h) + (y - k)(y₁ - k) = r²

**En resumen, la recta tangente es la línea que "toca" la circunferencia en un solo punto, formando un ángulo de 90 grados con el radio en ese punto.**

¿Te gustaría que te diera un ejemplo más concreto o que te explicara cómo encontrar la ecuación de una recta tangente a una circunferencia?


### CON RAG
Según el texto proporcionado, una recta tangente a una circunferencia es una recta que toca la circunferencia en un único punto.  La información clave sobre esto se encuentra en las secciones sobre mediatrices y circunferencias.  Específicamente:

*   **Definición de tangente:**  La definición de tangente se relaciona con la distancia desde un punto de la circunferencia al centro.
*   **Ejemplo de mediatriz:** El cálculo de la mediatriz de un segmento AB que pasa por un punto en la circunferencia demuestra cómo las rectas tangentes se definen en relación con la circunferencia y su centro.
*   **Ejemplo de hipérbola:** La definición de una hipérbola como el lugar geométrico de puntos equidistantes de un foco y una directriz también se relaciona con la tangente.

En resumen, la recta tangente es una línea que solo toca la circunferencia en un punto.

## 7. Evaluación CON vs. SIN RAG

### 7.1 Diseño

Se usan **11 preguntas** de tres tipos, elegidas para que la comparación muestre tanto las ventajas como los límites del RAG:

| Tipo | Cant. | Qué se espera |
|---|---|---|
| **Específica** del apunte (notación, recursos didácticos y procedimientos propios del curso) | 8 | El RAG debería superar claramente al modelo solo |
| **General** (conceptos que el modelo ya conoce) | 2 | Diferencia chica: sirve como control |
| **Fuera del corpus** | 1 | El RAG debería decir que no tiene información; el modelo solo responde igual |

Para cada pregunta se registran:

- **Evidencia:** una o más frases que **solo** aparecen en el fragmento que responde la pregunta. Si algún chunk recuperado las contiene todas, el retriever "acertó". Esto permite medir la **recuperación** de forma automática y separarla de la calidad de la **generación**.
- **Respuesta de referencia:** lo que dice el apunte, para corregir a mano las respuestas.

In [12]:
PREGUNTAS = [
    {"id": "P1", "tipo": "específica",
     "pregunta": "¿Qué propiedad cumple la recta tangente a una circunferencia respecto del radio en el punto de tangencia?",
     "evidencia": ["es tangente a la circunferencia"],
     "referencia": "La recta que pasa por el punto de tangencia y el centro es perpendicular a la tangente; "
                   "por eso la pendiente de la tangente es −1/m, siendo m la pendiente del radio (Plano cartesiano, sección 7)."},
    {"id": "P2", "tipo": "específica",
     "pregunta": "¿Qué son las condiciones iniciales y cuándo hay que explicitarlas?",
     "evidencia": ["condiciones iniciales", "inverso multiplicativo"],
     "referencia": "Son las condiciones que deben cumplir las variables para que la expresión tenga sentido "
                   "(divisor ≠ 0, radicando ≥ 0 en raíces de índice par). Deben escribirse en cada ejercicio, antes de operar (Números reales)."},
    {"id": "P3", "tipo": "específica",
     "pregunta": "¿En qué orden conviene escribir los valores de verdad al armar la tabla de dos proposiciones?",
     "evidencia": ["dos verdaderos y dos falsos"],
     "referencia": "Para la primera proposición: V, V, F, F. Para la segunda: alternar V y F empezando por V (Lógica, sección 3.2)."},
    {"id": "P4", "tipo": "específica",
     "pregunta": "¿Qué diferencia hay entre el conjunto vacío y el conjunto que tiene como único elemento al vacío? ¿Qué analogía usa el apunte?",
     "evidencia": ["bolsa vacia"],
     "referencia": "∅ no tiene elementos; {∅} tiene un elemento. Analogía: una bolsa vacía frente a una bolsa con otra bolsa vacía adentro (Conjuntos)."},
    {"id": "P5", "tipo": "específica",
     "pregunta": "¿Cómo se construye la tabla de seno, coseno y tangente de los ángulos especiales sin memorizarla?",
     "evidencia": ["enteros del 0 al 4"],
     "referencia": "En la fila del seno se escriben 0, 1, 2, 3, 4, se toma la raíz cuadrada y se divide por 2; "
                   "el coseno es la misma fila en orden inverso y la tangente es seno/coseno (Trigonometría, 5.1)."},
    {"id": "P6", "tipo": "específica",
     "pregunta": "¿Cuáles son los pasos de una demostración por inducción según el apunte?",
     "evidencia": ["caso base", "hipotesis inductiva"],
     "referencia": "1) Caso base: probar p(a). 2) Hipótesis inductiva: suponer p(h) verdadera. "
                   "3) Paso inductivo (tesis inductiva): probar p(h+1) usando la HI (Demostraciones, sección 5)."},
    {"id": "P7", "tipo": "específica",
     "pregunta": "¿En qué caso se puede usar la regla de Ruffini y qué devuelve?",
     "evidencia": ["ruffini", "divisor sea de la forma"],
     "referencia": "Cuando el divisor es de la forma (x − a), con a complejo. Da los coeficientes del cociente y el resto, "
                   "que por el teorema del resto vale P(a) (Expresiones algebraicas polinómicas, 5.2)."},
    {"id": "P8", "tipo": "específica",
     "pregunta": "¿Cómo se clasifican los sistemas de ecuaciones lineales según su conjunto solución?",
     "evidencia": ["compatible determinado", "incompatible"],
     "referencia": "Incompatible (sin solución), compatible determinado (solución única) y compatible indeterminado (infinitas soluciones)."},
    {"id": "P9", "tipo": "general",
     "pregunta": "¿Qué es un número complejo?",
     "evidencia": ["pares ordenados", "parte real"],
     "referencia": "Un par ordenado (a; b) de números reales, con parte real a y parte imaginaria b; en forma binómica a + bi, con i² = −1."},
    {"id": "P10", "tipo": "general",
     "pregunta": "¿Qué condición debe cumplir una función para tener función inversa?",
     "evidencia": ["biyectiva", "funcion inversa"],
     "referencia": "Debe ser biyectiva (inyectiva y sobreyectiva); si no lo es, se puede restringir dominio y codominio (Funciones, sección 5)."},
    {"id": "P11", "tipo": "fuera de corpus",
     "pregunta": "¿Cómo se calcula la derivada de un producto de funciones?",
     "evidencia": None,
     "referencia": "El tema no está en los apuntes. Respuesta esperada CON RAG: indicar que el contexto no alcanza."},
]
pd.DataFrame(PREGUNTAS)[["id", "tipo", "pregunta"]]

,id,tipo,pregunta
0,P1,específica,¿Qué propiedad cumple la recta tangente a una circunferencia respecto del radio en el punto de tangencia?
1,P2,específica,¿Qué son las condiciones iniciales y cuándo hay que explicitarlas?
2,P3,específica,¿En qué orden conviene escribir los valores de verdad al armar la tabla de dos proposiciones?
3,P4,específica,¿Qué diferencia hay entre el conjunto vacío y el conjunto que tiene como único elemento al vacío? ¿Qué analogía usa ...
4,P5,específica,"¿Cómo se construye la tabla de seno, coseno y tangente de los ángulos especiales sin memorizarla?"
5,P6,específica,¿Cuáles son los pasos de una demostración por inducción según el apunte?
6,P7,específica,¿En qué caso se puede usar la regla de Ruffini y qué devuelve?
7,P8,específica,¿Cómo se clasifican los sistemas de ecuaciones lineales según su conjunto solución?
8,P9,general,¿Qué es un número complejo?
9,P10,general,¿Qué condición debe cumplir una función para tener función inversa?


### 7.2 Ejecución

Para cada pregunta se recupera el contexto, se generan ambas respuestas y se calcula en qué posición del top-k apareció el chunk con la evidencia (`rank_evidencia`; vacío si no apareció). Los resultados se guardan en `resultados/evaluacion_respuestas.csv`.

In [13]:
def rank_evidencia(contexto: list[dict], evidencia) -> int | None:
    # Posición (1..k) del primer chunk que contiene TODAS las frases de evidencia.
    if not evidencia:
        return None
    for i, c in enumerate(contexto, start=1):
        texto = normalizar(c["text"])
        if all(normalizar(frase) in texto for frase in evidencia):
            return i
    return None


resultados = []
for p in PREGUNTAS:
    t0 = time.time()
    ctx = recuperar(collection, p["pregunta"])
    resultados.append({
        "id": p["id"], "tipo": p["tipo"], "pregunta": p["pregunta"],
        "rank_evidencia": rank_evidencia(ctx, p["evidencia"]),
        "fuentes_top3": "; ".join(f"{c['meta']['source']} p.{c['meta']['page']}" for c in ctx[:3]),
        "sin_rag": responder_sin_rag(p["pregunta"]),
        "con_rag": responder_con_rag(p["pregunta"], ctx),
        "referencia": p["referencia"],
    })
    print(f"{p['id']} listo ({time.time() - t0:.0f}s)")

df = pd.DataFrame(resultados)
df.to_csv(RESULTADOS_DIR / "evaluacion_respuestas.csv", index=False, encoding="utf-8")
df[["id", "tipo", "rank_evidencia", "fuentes_top3"]]

P1 listo (12s)
P2 listo (36s)
P3 listo (23s)
P4 listo (26s)
P5 listo (47s)
P6 listo (26s)
P7 listo (37s)
P8 listo (33s)
P9 listo (37s)
P10 listo (31s)
P11 listo (44s)


,id,tipo,rank_evidencia,fuentes_top3
0,P1,específica,9.0,T08. Plano cartesiano.pdf p.45; T08. Plano cartesiano.pdf p.47; T08. Plano cartesiano.pdf p.49
1,P2,específica,3.0,T01. Lógica Proposicional.pdf p.11; T01. Lógica Proposicional.pdf p.7; T03. Números reales.pdf p.19
2,P3,específica,5.0,T01. Lógica Proposicional.pdf p.5; T01. Lógica Proposicional.pdf p.15; T01. Lógica Proposicional.pdf p.19
3,P4,específica,1.0,T02. Conjuntos.pdf p.13; T09. Funciones.pdf p.3; T02. Conjuntos.pdf p.3
4,P5,específica,2.0,T11. Trigonometría.pdf p.9; T11. Trigonometría.pdf p.18; T11. Trigonometría.pdf p.25
5,P6,específica,5.0,T04. Demostraciones.pdf p.2; T04. Demostraciones.pdf p.3; T04. Demostraciones.pdf p.10
6,P7,específica,2.0,T06. Expresiones Algebraicas Polinómicas.pdf p.25; T06. Expresiones Algebraicas Polinómicas.pdf p.20; T06. Expresion...
7,P8,específica,NaN,T13. Sistemas (1C 2026).pdf p.2; T07. Exp Alg no polinómicas. Ecuaciones e Inecuaciones.pdf p.6; T13. Sistemas (1C 2...
8,P9,general,6.0,T05. Números Complejos.pdf p.2; T05. Números Complejos.pdf p.6; T05. Números Complejos.pdf p.7
9,P10,general,2.0,T09. Funciones.pdf p.63; T11. Trigonometría.pdf p.44; T09. Funciones.pdf p.63


In [14]:
# Respuestas lado a lado, para leer y corregir
for r in resultados:
    display(Markdown(
        f"---\n### {r['id']} ({r['tipo']}) — {r['pregunta']}\n\n"
        f"**Referencia:** {r['referencia']}\n\n"
        f"**Evidencia recuperada en la posición:** {r['rank_evidencia'] or '—'}\n\n"
        f"#### SIN RAG\n{r['sin_rag']}\n\n#### CON RAG\n{r['con_rag']}"
    ))

---
### P1 (específica) — ¿Qué propiedad cumple la recta tangente a una circunferencia respecto del radio en el punto de tangencia?

**Referencia:** La recta que pasa por el punto de tangencia y el centro es perpendicular a la tangente; por eso la pendiente de la tangente es −1/m, siendo m la pendiente del radio (Plano cartesiano, sección 7).

**Evidencia recuperada en la posición:** 9

#### SIN RAG
La recta tangente a una circunferencia en el punto de tangencia es **perpendicular** al radio en ese punto.

**Explicación:**

* **Circunferencia:** Es la figura geométrica formada por todos los puntos que están a la misma distancia de un punto fijo llamado centro.
* **Radio:** Es la distancia desde el centro de la circunferencia hasta cualquier punto de la circunferencia.
* **Tangente:** Es una línea recta que toca la circunferencia en un solo punto (el punto de tangencia) y no la corta.

En el punto de tangencia, la recta tangente forma un ángulo de 90 grados (un ángulo recto) con el radio.  Esto es una propiedad fundamental de la geometría y se basa en la definición de tangente y radio.

**En resumen:**  La recta tangente es perpendicular al radio en el punto de tangencia.


#### CON RAG
La recta tangente a una circunferencia es perpendicular al radio en el punto de tangencia.

---
### P2 (específica) — ¿Qué son las condiciones iniciales y cuándo hay que explicitarlas?

**Referencia:** Son las condiciones que deben cumplir las variables para que la expresión tenga sentido (divisor ≠ 0, radicando ≥ 0 en raíces de índice par). Deben escribirse en cada ejercicio, antes de operar (Números reales).

**Evidencia recuperada en la posición:** 3

#### SIN RAG
Las condiciones iniciales son un concepto fundamental en diversos campos, especialmente en:

* **Matemáticas:** Se refieren a los valores de una función o ecuación en un punto específico (o en un conjunto de puntos) al inicio de un proceso.  Piensa en una función como una montaña rusa: las condiciones iniciales son la altura y velocidad de la montaña rusa en el punto de inicio.
* **Física:**  En la resolución de ecuaciones diferenciales, las condiciones iniciales especifican el estado del sistema físico (posición, velocidad, etc.) en un momento particular.  Por ejemplo, si estás modelando el movimiento de un objeto, necesitas saber su posición y velocidad al principio.
* **Ingeniería:**  En el diseño y análisis de sistemas, las condiciones iniciales definen el estado inicial del sistema, lo que es crucial para predecir su comportamiento futuro.
* **Economía y Finanzas:**  En modelos económicos, las condiciones iniciales pueden representar el nivel de ingresos, la inflación, el desempleo, etc., al comienzo de un período.

**¿Cuándo hay que explicitarlas?**

La necesidad de explicitar las condiciones iniciales depende del problema que estés resolviendo:

1. **Ecuaciones Diferenciales:**  **Siempre** son necesarias. Las ecuaciones diferenciales describen cómo una función cambia con el tiempo o con respecto a otra variable.  Sin condiciones iniciales, la ecuación diferencial tiene infinitas soluciones posibles. Las condiciones iniciales actúan como "anclas" que restringen el espacio de soluciones y permiten encontrar la solución *única* que satisface la ecuación en un punto específico.  Por ejemplo:

   *  `dy/dx = f(x, y)`  con condiciones iniciales `y(x₀) = y₀` (valor de y en x₀) y `dy/dx(x₀) = y₀'` (derivada de y en x₀).

2. **Modelos Matemáticos en General:**  Cuando se construye un modelo matemático para representar un sistema, las condiciones iniciales son esenciales para:
   * **Simulaciones:**  Para que una simulación sea realista, necesitas saber dónde estaba el sistema al principio.
   * **Análisis de Sensibilidad:**  Para entender cómo las condiciones iniciales afectan la solución final.
   * **Validación del Modelo:**  Para comparar la salida del modelo con datos reales y verificar su precisión.

3. **Problemas de Optimización:**  En problemas de optimización, las condiciones iniciales pueden ser los valores iniciales de las variables de decisión que se están optimizando.

4. **Problemas de Programación Lineal/No Lineal:**  Las condiciones iniciales pueden ser los valores iniciales de las variables de la función objetivo y las restricciones.

**En resumen:**

* **Si estás resolviendo una ecuación diferencial, las condiciones iniciales son *obligatorias*.**
* **Si estás construyendo un modelo matemático, las condiciones iniciales son *cruciales* para obtener una solución significativa y útil.**
* **Si estás simulando un sistema, las condiciones iniciales son *esenciales* para la simulación.**

**Ejemplo sencillo (Ecuación Diferencial):**

Considera la ecuación diferencial: `dy/dx = y` con la condición inicial `y(0) = 1`.  Esta ecuación describe una función que crece exponencialmente.  La condición inicial `y(0) = 1` nos dice que la función comienza en el valor 1 en el punto x=0.  Con esta condición, podemos encontrar la solución general de la ecuación: `y(x) = e^x`.

Espero que esta explicación sea clara.  Si tienes alguna otra pregunta, no dudes en preguntar.


#### CON RAG
Según los documentos proporcionados, las condiciones iniciales son las premisas o datos que se necesitan para resolver un problema o analizar una situación. En el contexto de los ejemplos, se mencionan:

*   **En la ecuación canónica de una circunferencia:** Se necesitan encontrar puntos que pertenecen a la curva y determinar si un punto pertenece o no a esa curva.
*   **En la lógica proposicional:** Se identifican y nombran las proposiciones simples (t, w, p, q, r) que componen una proposición compuesta.
*   **En la inclusión de conjuntos:** Se necesita determinar si un conjunto A está incluido en B pero son distintos (A ⊊B).

**Cuándo hay que explicitarlas:**

Se deben explicitar cuando se está trabajando con un problema que requiere un análisis formal, como en el caso de la ecuación canónica de una circunferencia o la lógica proposicional.  En estos casos, es necesario definir claramente las variables, los valores y las relaciones entre ellas para poder realizar los cálculos y llegar a una conclusión.  En el ejemplo de la inclusión de conjuntos, se nombran y definen las proposiciones componentes para poder analizar la estructura de la proposición compuesta.


---
### P3 (específica) — ¿En qué orden conviene escribir los valores de verdad al armar la tabla de dos proposiciones?

**Referencia:** Para la primera proposición: V, V, F, F. Para la segunda: alternar V y F empezando por V (Lógica, sección 3.2).

**Evidencia recuperada en la posición:** 5

#### SIN RAG
La forma más eficiente y clara de armar una tabla de verdad para dos proposiciones es seguir un orden sistemático. Aquí te presento el orden recomendado y la lógica detrás de él:

**1. Escribe todas las combinaciones posibles de valores de verdad para las dos proposiciones.**

   * **Proposición A:** Tiene dos posibles valores de verdad: Verdadero (V) o Falso (F).
   * **Proposición B:** También tiene dos posibles valores de verdad: Verdadero (V) o Falso (F).

   Esto significa que hay 2 * 2 = 4 combinaciones posibles.

**2. Organiza las combinaciones en una tabla de 2x2.**

   | A     | B     |
   |-------|-------|
   | V     | V     |
   | V     | F     |
   | F     | V     |
   | F     | F     |

**3. Asigna los valores de verdad a cada celda de la tabla.**

   *  Comienza en la esquina superior izquierda (celda 1) y llena la tabla fila por fila, columna por columna.
   *  Asigna el valor de verdad (V o F) de la proposición A a la columna correspondiente.
   *  Asigna el valor de verdad (V o F) de la proposición B a la fila correspondiente.

**Por qué este orden es el mejor:**

* **Sistemático:**  Asegura que todas las combinaciones posibles se consideren.
* **Claro:** Facilita la visualización de la relación entre las proposiciones y sus valores de verdad.
* **Evita errores:**  Al seguir un orden predefinido, reduces la posibilidad de olvidar alguna combinación o asignar valores de verdad incorrectamente.

**Ejemplo:**

Para la tabla anterior, cada celda ya tiene asignado el valor de verdad correspondiente a las proposiciones A y B.  No hay un orden específico para *escribir* los valores dentro de cada celda, ya que ya están predeterminados por la combinación de filas y columnas.

**En resumen, el orden clave es primero listar todas las combinaciones posibles y luego organizarlas en una tabla de 2x2, asignando los valores de verdad de manera sistemática.**


#### CON RAG
Al armar la tabla de verdad de dos proposiciones, conviene escribir los valores de verdad en las columnas de ambas proposiciones, **simétricamente**. Esto significa que la primera columna será para la proposición 1, la segunda columna para la proposición 2, y así sucesivamente.  

Esto permite que al evaluar cada combinación de valores de verdad (las filas de la tabla), puedas ver directamente el valor de verdad de la proposición compuesta (el resultado de la operación lógica que las conecta) para cada posible combinación.


---
### P4 (específica) — ¿Qué diferencia hay entre el conjunto vacío y el conjunto que tiene como único elemento al vacío? ¿Qué analogía usa el apunte?

**Referencia:** ∅ no tiene elementos; {∅} tiene un elemento. Analogía: una bolsa vacía frente a una bolsa con otra bolsa vacía adentro (Conjuntos).

**Evidencia recuperada en la posición:** 1

#### SIN RAG
¡Excelente pregunta! La diferencia entre el conjunto vacío y el conjunto que tiene como único elemento al vacío puede ser sutil, pero es importante.

**1. El Conjunto Vacío (∅ o {})**

*   El conjunto vacío es un conjunto que **no contiene ningún elemento**. Es decir, es un conjunto que está "vacío" de elementos.
*   Es un conjunto bien definido, lo que significa que tiene una regla clara que determina qué elementos pertenecen a él (en este caso, ninguna).
*   Es un conjunto fundamental en la teoría de conjuntos y se utiliza como base para muchas definiciones y demostraciones.

**2. El Conjunto que Tiene como Único Elemento al Vacío ({∅})**

*   Este conjunto contiene **solo un elemento**, y ese elemento es el conjunto vacío en sí mismo.
*   Es un conjunto que es un poco más "complejo" porque contiene un conjunto como elemento.
*   Aunque es válido, este conjunto es menos común y a menudo se considera una construcción más artificial.

**En resumen:**

| Característica       | Conjunto Vacío (∅) | Conjunto con {∅} |
| --------------------- | ------------------ | ---------------- |
| Elementos             | Ninguno            | Uno (el conjunto vacío) |
| Naturaleza            | Fundamental        | Más complejo     |
| Uso común             | Muy frecuente      | Menos frecuente  |


**Analogía del Apunte:**

El apunte, que es un cuaderno de apuntes, utiliza una analogía muy útil para entender esta diferencia.  El apunte dice algo así:

"**El conjunto vacío es como un cuaderno de apuntes que no tiene ninguna página.**  Es un cuaderno que no contiene nada.  El conjunto que tiene como único elemento al vacío es como un cuaderno de apuntes que tiene una sola página, y esa página es un cuaderno de apuntes que no tiene ninguna página."

La analogía destaca que ambos representan la ausencia de contenido, pero el segundo (con {∅}) introduce una capa adicional de "contenedores" vacíos.

**En esencia, el conjunto vacío es la ausencia de elementos, mientras que el conjunto con {∅} es la ausencia de elementos dentro de un contenedor que también es la ausencia de elementos.**

Espero que esta explicación y la analogía del apunte te ayuden a comprender mejor la diferencia. ¡No dudes en preguntar si tienes más dudas!

#### CON RAG
Según el apunte, la diferencia entre el conjunto vacío (∅) y el conjunto que tiene como único elemento al vacío ({∅}) radica en que el primero es un conjunto que no contiene ningún elemento, mientras que el segundo contiene un solo elemento, que es precisamente el conjunto vacío.

La analogía que utiliza el apunte para ilustrar esta diferencia es la de una compra de supermercado:

*   **Conjunto vacío (∅):**  Sería como llevar un carrito vacío a la caja. No hay nada en el carrito, por lo tanto, no hay elementos en el conjunto.
*   **Conjunto con {∅} como elemento:** Sería como llevar un paquete vacío a la caja. El paquete en sí mismo es el elemento del conjunto, y ese elemento es el conjunto vacío.

En resumen, el apunte busca hacer la idea más tangible y comprensible a través de una situación cotidiana.

---
### P5 (específica) — ¿Cómo se construye la tabla de seno, coseno y tangente de los ángulos especiales sin memorizarla?

**Referencia:** En la fila del seno se escriben 0, 1, 2, 3, 4, se toma la raíz cuadrada y se divide por 2; el coseno es la misma fila en orden inverso y la tangente es seno/coseno (Trigonometría, 5.1).

**Evidencia recuperada en la posición:** 2

#### SIN RAG
Construir la tabla de seno, coseno y tangente de los ángulos especiales sin memorizarla es un ejercicio de comprensión y manipulación de relaciones trigonométricas. Aquí te explico el proceso paso a paso:

**1. Entendiendo las Relaciones Trigonométricas Básicas:**

* **Triángulo Rectángulo y Ángulos Especiales:**  Comienza con un triángulo rectángulo. Los ángulos especiales que te interesan son 30°, 45° y 60°.  En estos triángulos, las razones de los lados (catetos y hipotenusa) se convierten en las funciones trigonométricas.
* **Definiciones:**
    * **Seno (sin θ):** Cateto opuesto / Hipotenusa
    * **Coseno (cos θ):** Cateto adyacente / Hipotenusa
    * **Tangente (tan θ):** Cateto opuesto / Cateto adyacente

**2. Construyendo los Triángulos Rectángulos Especiales:**

* **30° - 60° - 90°:**
    * **30°:**  El ángulo de 30° es el ángulo agudo.  El lado opuesto al ángulo de 30° es la base (l = 1).  El lado adyacente al ángulo de 30° es la altura (w = √3).  La hipotenusa es 2.
    * **60°:**  El ángulo de 60° es el ángulo agudo.  El lado opuesto al ángulo de 60° es la altura (w = √3).  El lado adyacente al ángulo de 60° es la base (l = 1).  La hipotenusa es 2.
* **45° - 45° - 90°:**
    * **45°:**  Este es un triángulo isósceles.  Los dos catetos son iguales (l = w).  La hipotenusa es √2.  Puedes usar el teorema de Pitágoras (a² + b² = c²) para encontrar la hipotenusa:  1² + 1² = √2.
* **60° - 60° - 90°:**
    * **60°:**  Similar al triángulo de 30° - 60° - 90°, pero con los lados invertidos.

**3. Calculando los Valores Trigonométricos:**

Ahora que tienes los triángulos especiales, puedes calcular los valores de seno, coseno y tangente para cada ángulo:

* **30°:**
    * sin(30°) = 1/2
    * cos(30°) = √3/2
    * tan(30°) = 1/√3 = √3/3
* **45°:**
    * sin(45°) = 1/√2 = √2/2
    * cos(45°) = 1/√2 = √2/2
    * tan(45°) = 1
* **60°:**
    * sin(60°) = √3/2
    * cos(60°) = 1/2
    * tan(60°) = √3

**4.  Extrapolando y Generalizando (Sin Memorizar):**

* **La clave está en las relaciones de los lados:**  En lugar de memorizar los valores, comprende cómo se relacionan.  Por ejemplo, en un triángulo 30-60-90, el lado más corto (opuesto al 30°) es 1, el lado más largo (opuesto al 60°) es √3, y la hipotenusa es 2.
* **Usa el Teorema de Pitágoras:**  a² + b² = c²  Siempre que tengas un triángulo rectángulo, puedes usar esta ecuación para encontrar las longitudes de los lados.
* **Manipula las Razones:**  Puedes usar las definiciones de seno, coseno y tangente para encontrar otros valores.  Por ejemplo, si conoces la longitud de un lado y el ángulo, puedes calcular la longitud de otro lado y luego usar las definiciones para encontrar las razones trigonométricas.

**Ejemplo de cómo construir la tabla:**

| Ángulo (θ) | Cateto Opuesto (a) | Cateto Adyacente (b) | Hipotenusa (c) | sin(θ) | cos(θ) | tan(θ) |
|---|---|---|---|---|---|---|
| 30° | 1 | √3 | 2 | 1/2 | √3/2 | √3/3 |
| 45° | 1 | 1 | √2 | √2/2 | √2/2 | 1 |
| 60° | √3 | 1 | 2 | √3/2 | 1/2 | √3 |

**En resumen:**  En lugar de memorizar los valores, construye la tabla a partir de los triángulos especiales, calcula los valores usando las definiciones trigonométricas y el teorema de Pitágoras, y comprende las relaciones entre los lados y las funciones trigonométricas.  Esto te permitirá construir la tabla y calcular valores para cualquier ángulo especial.


#### CON RAG
La tabla de seno, coseno y tangente de los ángulos especiales se construye siguiendo estos pasos:

1.  **Ubicación de los ángulos:** Se ubican los ángulos especiales (0, π/6, π/4, π/3, π/2, etc.) en una circunferencia trigonométrica.

2.  **Columna del seno:** Se escribe los enteros del 0 al 4 en la columna del seno.

3.  **Raíz cuadrada:** Se calcula la raíz cuadrada de cada uno de los enteros en la columna del seno.

4.  **División por 2:** Se divide la raíz cuadrada obtenida por 2.  El resultado es el seno del ángulo correspondiente.

5.  **Columna del coseno:** Se copia la columna del seno en orden contrario.  Esto se debe a que los ángulos son complementarios (por ejemplo, π/6 y 5π/6 son complementarios).

6.  **Columna de la tangente:** Se calcula la tangente de cada ángulo (seno dividido por coseno) y se escribe el resultado en la columna de la tangente.

En esencia, la tabla se construye utilizando la relación entre los ángulos y sus funciones trigonométricas, evitando la necesidad de memorizar los valores directamente.


...

---
### P7 (específica) — ¿En qué caso se puede usar la regla de Ruffini y qué devuelve?

**Referencia:** Cuando el divisor es de la forma (x − a), con a complejo. Da los coeficientes del cociente y el resto, que por el teorema del resto vale P(a) (Expresiones algebraicas polinómicas, 5.2).

**Evidencia recuperada en la posición:** 2

#### SIN RAG
La regla de Ruffini (también conocida como división sintética) es una forma abreviada de realizar la división de un polinomio por un binomio de la forma (x - a).  Aquí te explico cuándo se usa y qué devuelve:

**Cuándo se usa la regla de Ruffini:**

La regla de Ruffini se utiliza para dividir un polinomio de grado *n* por un binomio de la forma (x - a), donde *n* es el grado del polinomio.  En otras palabras, se usa cuando conoces una raíz del binomio (el valor de 'a' que hace que el binomio sea cero) y quieres encontrar los otros coeficientes del cociente y el resto.

**¿Qué devuelve?**

La regla de Ruffini devuelve dos cosas:

1. **El cociente:**  Es un polinomio de grado *n-1* (uno menos que el grado del polinomio original) que representa la división del polinomio original entre (x - a).

2. **El resto:** Es un número que representa el valor del polinomio original evaluado en 'a'.  En otras palabras, es el valor del polinomio cuando x = a.  El resto es crucial porque si el resto es cero, entonces 'a' es una raíz del polinomio original, y (x - a) es un factor del polinomio.

**Cómo funciona la regla de Ruffini (ejemplo):**

Supongamos que queremos dividir el polinomio  P(x) = x³ - 6x² + 11x - 6  por el binomio (x - 1).

La regla de Ruffini se ve así:

```
1 |  1  -6   11  -6
  |      1  -5   6
  ------------------
     1  -5   6   0
```

* **El primer número (1) en la fila superior** es el coeficiente principal del polinomio P(x).
* **El segundo número (-6) en la fila superior** es el coeficiente del término x².
* **Y así sucesivamente...**

**Interpretación de los resultados:**

* **La primera fila (1 -6 11 -6)** es la misma que el polinomio P(x) = x³ - 6x² + 11x - 6.
* **La segunda fila (-5 6)**  es el resultado de realizar operaciones aritméticas (multiplicación y suma) en la primera fila, utilizando el valor de 'a' (en este caso, 'a' = 1).
* **La tercera fila (1 -5 6 0)**  es el cociente y el resto.  En este caso:
    * **Cociente:**  x² - 5x + 6
    * **Resto:** 0

Esto significa que:  x³ - 6x² + 11x - 6 = (x - 1)(x² - 5x + 6)

**En resumen:** La regla de Ruffini es una herramienta eficiente para dividir polinomios cuando conoces una raíz del divisor.  Devuelve el cociente y el resto de la división.

**Recursos adicionales:**

* **Video explicativo:** [https://m.youtube.com/watch?v=9j-9j-9j-9j](https://m.youtube.com/watch?v=9j-9j-9j-9j)
* **Calculadora de Ruffini online:** [https://www.symbolab.com/solver/ruffini-algorithm-calculator](https://www.symbolab.com/solver/ruffini-algorithm-calculator)

Espero que esta explicación te sea útil.  ¡No dudes en preguntar si tienes más dudas!


#### CON RAG
La regla de Ruffini se utiliza para dividir un polinomio de grado 2 (un polinomio cuadrático) entre un binomio de la forma (x - a). 

**¿Qué devuelve?**

La regla de Ruffini devuelve un cociente (un polinomio de grado 1, también llamado lineal) y un resto.  La división se expresa así:

P(x) = (x - a) * Q(x) + R

Donde:

*   P(x) es el polinomio original de grado 2.
*   (x - a) es el binomio divisor.
*   Q(x) es el cociente (un polinomio lineal).
*   R es el resto (un número).

En resumen, la regla de Ruffini permite realizar la división de un polinomio cuadrático por un binomio de la forma (x - a) de manera eficiente, obteniendo el cociente y el resto.


---
### P8 (específica) — ¿Cómo se clasifican los sistemas de ecuaciones lineales según su conjunto solución?

**Referencia:** Incompatible (sin solución), compatible determinado (solución única) y compatible indeterminado (infinitas soluciones).

**Evidencia recuperada en la posición:** —

#### SIN RAG
Los sistemas de ecuaciones lineales se clasifican según su conjunto solución de las siguientes maneras:

**1. Según el número de soluciones:**

* **Sistema con una única solución (o sistema consistente determinado):**  El gráfico de las ecuaciones representa una línea recta única.  Esto significa que hay un punto de intersección único que satisface ambas ecuaciones.  Se puede expresar como una única solución en forma de coordenadas (x, y).

* **Sistema con infinitas soluciones (o sistema consistente indeterminado):** El gráfico de las ecuaciones representa una familia de líneas rectas que son todas equivalentes.  Esto significa que hay un número infinito de pares de valores (x, y) que satisfacen ambas ecuaciones.  Se expresa como una solución en forma de parámetros (x, y = f(t)).  En este caso, las ecuaciones son *dependientes*.

* **Sistema sin solución (o sistema inconsistente):** El gráfico de las ecuaciones representa dos o más líneas rectas que nunca se cruzan.  Esto significa que no hay ningún valor de x e y que satisfaga ambas ecuaciones simultáneamente.  Se puede expresar como "no hay solución".


**2. Según la naturaleza de las ecuaciones:**

* **Sistema consistente y determinado:**  Ambas ecuaciones representan la misma línea recta.  Esto significa que tienen la misma pendiente y el mismo punto de intersección.  La solución es una única solución.

* **Sistema consistente e indeterminado:**  Las ecuaciones representan la misma línea recta, pero una de ellas es un múltiplo del otro.  Esto significa que una ecuación puede ser obtenida de la otra mediante una operación algebraica (como suma o resta).  La solución es una familia de soluciones.

* **Sistema inconsistente:**  Las ecuaciones representan líneas rectas paralelas.  Esto significa que tienen la misma pendiente pero diferentes puntos de intersección.  No hay solución.

**Resumen en una tabla:**

| Característica          | Sistema Consistente Determinado | Sistema Consistente Indeterminado | Sistema Inconsistente |
|--------------------------|---------------------------------|------------------------------------|-----------------------|
| **Número de soluciones** | 1                               | Infinitas                          | 0                     |
| **Gráfica**              | Línea recta única                | Familia de líneas rectas paralelas | Líneas rectas paralelas |
| **Ecuaciones**           | Dependientes                     | Dependientes                        | Independientes        |
| **Conjunto solución**     | (x, y)                            | (x, y = f(t))                      | No hay solución       |



**Ejemplo:**

* **Sistema Consistente Determinado:**
   *  x + y = 5
   *  2x + 2y = 10
   La segunda ecuación es simplemente el doble de la primera.  La solución es x = 5, y = 0.

* **Sistema Consistente Indeterminado:**
   *  x + y = 3
   *  2x + 2y = 6
   Ambas ecuaciones representan la misma línea recta.  La solución es x = t, y = 3 - t, donde 't' es cualquier número real.

* **Sistema Inconsistente:**
   *  x + y = 1
   *  x + y = 2
   Las líneas son paralelas y nunca se cruzan.  No hay solución.

Espero que esta explicación detallada te sea útil.  ¿Tienes alguna otra pregunta o quieres que te explique algún aspecto con más detalle?


#### CON RAG
Los sistemas de ecuaciones lineales se clasifican según su conjunto solución de la siguiente manera:

*   **Sistema Compatible y Exactamente Soluble:** El sistema tiene una única solución.
*   **Sistema Compatible y con Infinitas Soluciones:** El sistema tiene un número infinito de soluciones. Esto ocurre cuando las ecuaciones son dependientes linealmente (es decir, una ecuación puede ser obtenida de la otra multiplicando una de ellas por una constante).
*   **Sistema Incompatible:** El sistema no tiene solución. Esto ocurre cuando las ecuaciones son inconsistentes (es decir, no pueden ser satisfejidas simultáneamente).

---
### P9 (general) — ¿Qué es un número complejo?

**Referencia:** Un par ordenado (a; b) de números reales, con parte real a y parte imaginaria b; en forma binómica a + bi, con i² = −1.

**Evidencia recuperada en la posición:** 6

#### SIN RAG
Un número complejo es una extensión del concepto de número real que permite trabajar con raíces cuadradas de números negativos.  Aquí te lo explico de forma detallada:

**1. Origen del Concepto:**

* **Problema de la raíz cuadrada de -1:**  En los números reales, no existe un número que, multiplicado por sí mismo, dé como resultado un número negativo.  Por ejemplo, 2 * 2 = 4, pero (-2) * (-2) = 4.  Esto crea un problema cuando intentamos encontrar la raíz cuadrada de números negativos.
* **Necesidad de una Extensión:**  Matemáticos del siglo XIX, como Carl Friedrich Gauss, reconocieron la necesidad de extender el sistema numérico para resolver problemas en áreas como la física y la ingeniería.

**2. Definición:**

Un número complejo se define como un número que tiene la forma:

**z = a + bi**

Donde:

* **a** y **b** son números reales.
* **i** es la **unidad imaginaria**, definida como la raíz cuadrada de -1:  **i = √(-1)**.  Es decir,  i² = -1.

* **a** se llama la **parte real** del número complejo z.
* **b** se llama la **parte imaginaria** del número complejo z.

**3. Ejemplos:**

* **3 + 2i:**  Aquí, a = 3 y b = 2.  Es un número complejo con parte real 3 y parte imaginaria 2.
* **-1 - i:** Aquí, a = -1 y b = -1. Es un número complejo con parte real -1 y parte imaginaria -1.
* **5i:**  Aquí, a = 0 y b = 5.  Es un número complejo donde la parte real es 0 (y se le llama un número imaginario puro).
* **7:**  Aquí, a = 7 y b = 0.  Es un número real, que también es un número complejo (ya que se puede escribir como 7 + 0i).

**4. Representación Gráfica:**

Los números complejos se pueden representar en un plano llamado **plano complejo** o **plano de Argand:**

* El eje horizontal (eje x) representa la parte real (a).
* El eje vertical (eje y) representa la parte imaginaria (b).
* Cada número complejo z = a + bi se representa como un punto en este plano con coordenadas (a, b).

**5. Operaciones con Números Complejos:**

Se pueden realizar operaciones aritméticas con números complejos, como suma, resta, multiplicación y división, siguiendo reglas específicas que involucran la unidad imaginaria 'i'.

* **Suma:** (a + bi) + (c + di) = (a + c) + (b + d)i
* **Resta:** (a + bi) - (c + di) = (a - c) + (b - d)i
* **Multiplicación:** (a + bi) * (c + di) = (ac - bd) + (ad + bc)i
* **División:**  Se multiplica el numerador y el denominador por el conjugado del denominador.

**En resumen, un número complejo es una extensión del sistema numérico real que permite trabajar con raíces cuadradas de números negativos, utilizando la unidad imaginaria 'i' para representar estas raíces.**

¿Te gustaría que profundizáramos en algún aspecto específico, como:

*   Las operaciones con números complejos?
*   La representación gráfica en el plano complejo?
*   Las aplicaciones de los números complejos en diferentes campos (física, ingeniería, etc.)?

#### CON RAG
Según los documentos proporcionados, un número complejo (z) se define como un par ordenado de números reales: **z = (a; b), donde a ∈R ∧ b ∈R**.  Esto significa que un número complejo se representa mediante una coordenada real (a) y otra coordenada real (b), y se anota como (Re(z); Im(z)), donde Re(z) es la componente real y Im(z) es la componente imaginaria.  En esencia, los números complejos son una extensión de los números reales utilizando la unidad imaginaria "i", aunque en esta definición, la unidad imaginaria no se menciona explícitamente, sino que se representa a través de la segunda componente del par ordenado.


---
### P10 (general) — ¿Qué condición debe cumplir una función para tener función inversa?

**Referencia:** Debe ser biyectiva (inyectiva y sobreyectiva); si no lo es, se puede restringir dominio y codominio (Funciones, sección 5).

**Evidencia recuperada en la posición:** 2

#### SIN RAG
Una función debe cumplir ciertas condiciones para tener una función inversa. Aquí te las detallo:

**1. Debe ser una función de una sola variable:**

*   La función inversa solo puede existir si la función original es de una sola variable (por ejemplo, f(x) = x + 2).  Si la función es de varias variables (por ejemplo, f(x, y) = x² + y), no puede tener una inversa que sea una función de una sola variable.

**2. Debe ser biyectiva (uno a uno y sobre):**

*   Esto significa que la función debe cumplir dos condiciones:

    *   **Inyectiva (Uno a uno):**  Cada elemento del dominio (la entrada de la función) debe corresponder a un único elemento del rango (la salida de la función).  En otras palabras, si f(x₁) = f(x₂), entonces x₁ = x₂.  Esto asegura que no haya valores diferentes en el dominio que produzcan el mismo valor en el rango.

    *   **Sobre:**  Cada elemento del rango debe corresponder a al menos un elemento del dominio.  Es decir, el rango debe tener la misma cardinalidad que el dominio.

**En resumen, una función para tener una función inversa debe ser una función de una sola variable que sea tanto inyectiva como sobre.**

**Cómo encontrar la función inversa:**

Si una función f(x) cumple estas condiciones, la función inversa se denota como f⁻¹(y) y se encuentra de la siguiente manera:

1.  Reescribe la función como y = f(x).
2.  Resuelve la ecuación para x en términos de y.  Esto te dará x = g(y), donde g(y) es la función inversa.
3.  Intercambia x e y:  f⁻¹(y) = g(y).

**Ejemplo:**

Considera la función f(x) = 2x + 1.

1.  y = 2x + 1
2.  Resuelve para x:  y - 1 = 2x  =>  x = (y - 1) / 2
3.  Intercambia x e y:  f⁻¹(y) = (y - 1) / 2

**Importante:**  No todas las funciones tienen una función inversa.  Solo las funciones biyectivas lo tienen.  Si una función no es biyectiva, no se puede encontrar una función inversa que sea también una función biyectiva.

Espero que esta explicación sea clara.  Si tienes alguna otra pregunta, no dudes en preguntar.

#### CON RAG
Según el texto proporcionado, una función debe ser biyectiva para tener una función inversa. Esto significa que la función debe ser tanto inyectiva (cada elemento del dominio corresponde a un único elemento del rango) como sobreyectiva (cada elemento del rango corresponde a al menos un elemento del dominio).

Además, el texto detalla las propiedades de la función inversa:

*   (f⁻¹ ◦ f)(x) = x para todo x ∈ A
*   (f ◦ f⁻¹) (y) = y para todo y ∈ B
*   Dom(f⁻¹) = Im(f) y Im(f⁻¹) = Dom(f)

En resumen, la clave para tener una función inversa es que la función original sea biyectiva.


---
### P11 (fuera de corpus) — ¿Cómo se calcula la derivada de un producto de funciones?

**Referencia:** El tema no está en los apuntes. Respuesta esperada CON RAG: indicar que el contexto no alcanza.

**Evidencia recuperada en la posición:** —

#### SIN RAG
La derivada de un producto de funciones se calcula utilizando la **regla del producto** en cálculo. Esta regla establece que la derivada de un producto de dos funciones, *u(x)* y *v(x)*, es:

**d/dx [u(x) * v(x)] = u'(x) * v(x) + u(x) * v'(x)**

En palabras:

* **u'(x) * v(x)**:  La derivada de la primera función *u(x)* multiplicada por la segunda función *v(x)*.
* **u(x) * v'(x)**: La primera función *u(x)* multiplicada por la derivada de la segunda función *v'(x)*.

**Explicación más detallada y ejemplos:**

**1. Caso General (Más de dos funciones):**

Si tienes un producto de más de dos funciones, puedes aplicar la regla del producto repetidamente.  Por ejemplo, si tienes:

d/dx [u(x) * v(x) * w(x)] =  [u'(x) * v(x) + u(x) * v'(x)] * w(x) + u(x) * [v'(x) * w(x) + v(x) * w'(x)]

**2. Ejemplos:**

* **Ejemplo 1:**  d/dx [x² * sin(x)]

   * u(x) = x²  =>  u'(x) = 2x
   * v(x) = sin(x) => v'(x) = cos(x)

   Aplicando la regla del producto:
   d/dx [x² * sin(x)] = (2x) * sin(x) + x² * cos(x)

* **Ejemplo 2:** d/dx [t³ * e^t]

   * u(t) = t³  =>  u'(t) = 3t²
   * v(t) = e^t  =>  v'(t) = e^t

   Aplicando la regla del producto:
   d/dx [t³ * e^t] = (3t²) * e^t + t³ * e^t

* **Ejemplo 3:** d/dx [f(x) * g(x) * h(x)]

   * d/dx [f(x) * g(x) * h(x)] = f'(x) * g(x) * h(x) + f(x) * [g'(x) * h(x) + g(x) * h'(x)] + f(x) * g(x) * h'(x)

**Puntos Clave:**

* **Identifica u(x) y v(x):**  Primero, identifica las dos funciones que están siendo multiplicadas.
* **Calcula las derivadas:** Calcula la derivada de u(x) (u'(x)) y la derivada de v(x) (v'(x)).
* **Aplica la regla:**  Sustituye las derivadas en la fórmula:  u'(x) * v(x) + u(x) * v'(x)

**En resumen, la regla del producto es una herramienta fundamental en cálculo para encontrar la derivada de productos de funciones.  Recuerda que la clave está en identificar las funciones y sus derivadas correctamente.**


#### CON RAG
La derivada de un producto de funciones se calcula utilizando la **regla del producto** del cálculo diferencial.  Esta regla establece que la derivada de un producto de dos funciones, *f(x)* y *g(x)*, es:

**d/dx [f(x)g(x)] = f'(x)g(x) + f(x)g'(x)**

En palabras:

*   **f'(x)g(x)**:  La derivada de la primera función multiplicada por la segunda función.
*   **f(x)g'(x)**:  La primera función multiplicada por la derivada de la segunda función.

**Ejemplo:**

Si tenemos  *y = (x² + 1)(3x - 2)*,  para encontrar *dy/dx*, aplicamos la regla del producto:

1.  **f(x) = x² + 1**  =>  *f'(x) = 2x*
2.  **g(x) = 3x - 2**  =>  *g'(x) = 3*

Entonces:

*dy/dx = (2x)(3x - 2) + (x² + 1)(3)*
*dy/dx = 6x² - 4x + 3x² + 3*
*dy/dx = 9x² - 4x + 3*

Por lo tanto, la derivada de *y* con respecto a *x* es *9x² - 4x + 3*.

**En resumen, la regla del producto es fundamental para encontrar la derivada de funciones que están multiplicadas entre sí.**


### 7.3 Corrección

La calidad de las respuestas se corrige **a mano** contra la referencia, con esta rúbrica:

| Columna | Valores | Criterio |
|---|---|---|
| `nota_sin_rag`, `nota_con_rag` | 0 / 0.5 / 1 | 1 = correcta y completa según el apunte; 0.5 = parcialmente correcta o genérica; 0 = incorrecta |
| `alucina_sin_rag`, `alucina_con_rag` | 0 / 1 | 1 si afirma algo falso o no sustentado (en P11, si responde en lugar de abstenerse) |
| `cita_fuente` | 0 / 1 | 1 si la respuesta CON RAG cita archivo y página correctos |

La celda siguiente crea `resultados/evaluacion_corregida.csv` con esas columnas vacías (si todavía no existe). Completala con Excel o LibreOffice y volvé a correr la celda de resumen.

In [15]:
ruta_corregida = RESULTADOS_DIR / "evaluacion_corregida.csv"
if not ruta_corregida.exists():
    plantilla = df.copy()
    for col in ["nota_sin_rag", "nota_con_rag", "alucina_sin_rag", "alucina_con_rag", "cita_fuente"]:
        plantilla[col] = ""
    plantilla.to_csv(ruta_corregida, index=False, encoding="utf-8")
    print(f"Plantilla creada: {ruta_corregida}. Completala y volvé a correr la celda siguiente.")
else:
    print(f"Ya existe {ruta_corregida}; no se sobrescribe.")

Plantilla creada: /Users/cristianczubara/MCD/CD.10 Seminario Intensivo de Tópicos Avanzados en Datos Complejos/curso-llm/tp_rag_matematica/resultados/evaluacion_corregida.csv. Completala y volvé a correr la celda siguiente.


In [20]:
# Resumen de la evaluación (requiere la planilla corregida)
ruta_corregida = RESULTADOS_DIR / "evaluacion_corregida.csv"
print("Leyendo:", ruta_corregida)

# Detecta la codificación y el separador (Numbers/Excel en español usan ";")
for codificacion in ("utf-8-sig", "latin-1"):
    try:
        with open(ruta_corregida, encoding=codificacion) as f:
            encabezado = f.readline()
        break
    except UnicodeDecodeError:
        continue
separador = ";" if encabezado.count(";") > encabezado.count(",") else ","
print(f"Separador detectado: '{separador}' | codificación: {codificacion}")

corr = pd.read_csv(ruta_corregida, sep=separador, encoding=codificacion)

cols = ["nota_sin_rag", "nota_con_rag", "alucina_sin_rag", "alucina_con_rag", "cita_fuente"]
faltan = [c for c in cols if c not in corr.columns]

if faltan:
    print("No encuentro estas columnas:", faltan)
    print("Columnas leídas:", list(corr.columns))
else:
    for c in cols:  # acepta "0,5" y "0.5"
        corr[c] = pd.to_numeric(corr[c].astype(str).str.replace(",", ".").str.strip(), errors="coerce")
    display(corr[["id", "tipo"] + cols])
    if corr[cols].isna().all().all():
        print("Las columnas de corrección siguen vacías: revisá dónde guardaste el archivo.")
    else:
        por_tipo = corr.groupby("tipo")[cols].mean().round(2)
        total = corr[cols].mean().round(2).rename("TOTAL")
        display(pd.concat([por_tipo, total.to_frame().T]))

Leyendo: /Users/cristianczubara/MCD/CD.10 Seminario Intensivo de Tópicos Avanzados en Datos Complejos/curso-llm/tp_rag_matematica/resultados/evaluacion_corregida.csv
Separador detectado: ';' | codificación: utf-8-sig


,id,tipo,nota_sin_rag,nota_con_rag,alucina_sin_rag,alucina_con_rag,cita_fuente
0,P1,específica,6,10,NaN,NaN,10
1,P2,específica,9,4,NaN,NaN,10
2,P3,específica,8,10,NaN,NaN,10
3,P4,específica,8,10,NaN,NaN,10
4,P5,específica,6,10,NaN,NaN,10
5,P6,específica,10,10,NaN,NaN,10
6,P7,específica,8,10,NaN,NaN,10
7,P8,específica,7,10,NaN,NaN,10
8,P9,general,7,10,NaN,NaN,10
9,P10,general,7,10,NaN,NaN,10


,nota_sin_rag,nota_con_rag,alucina_sin_rag,alucina_con_rag,cita_fuente
específica,7.75,9.25,NaN,NaN,10.0
fuera de corpus,8.00,0.00,NaN,NaN,10.0
general,7.00,10.00,NaN,NaN,10.0
TOTAL,7.64,8.55,NaN,NaN,10.0


## 8. Experimentos

### 8.1 Sensibilidad a *k*

Como ya se recuperaron 10 chunks por pregunta y se conoce la posición de la evidencia, se puede calcular sin costo extra la proporción de preguntas en las que la evidencia está entre los primeros *k* (**hit@k**) y el **MRR** (promedio de 1/posición). Esto muestra si `TOP_K = 10` es necesario o si con menos contexto alcanzaría.

In [21]:
def metricas_recuperacion(ranks: list) -> dict:
    # Cada rank es la posición (1..k) de la evidencia, o None si no apareció
    n = len(ranks)
    out = {f"hit@{k}": sum(1 for r in ranks if r and r <= k) / n for k in (1, 3, 5, 10)}
    out["MRR"] = sum(1 / r for r in ranks if r) / n
    return {k: round(v, 2) for k, v in out.items()}


con_evidencia = [p for p in PREGUNTAS if p["evidencia"]]
ranks_base = [r["rank_evidencia"] for r in resultados if r["tipo"] != "fuera de corpus"]
pd.DataFrame([metricas_recuperacion(ranks_base)], index=["sin prefijos (base)"])

,hit@1,hit@3,hit@5,hit@10,MRR
sin prefijos (base),0.1,0.5,0.7,0.9,0.35


### 8.2 Prefijos de tarea en `nomic-embed-text`

El modelo de embeddings fue entrenado para recibir `search_document: ` delante de cada fragmento indexado y `search_query: ` delante de cada consulta. Se construye una **segunda colección** con prefijos y se compara la recuperación sobre las mismas preguntas. Solo cambia el embedding; chunks, *k* y preguntas son los mismos.

> Reindexa todo el corpus en otra colección. Tarda lo mismo que la indexación original.

In [22]:
CORRER_EXPERIMENTO_PREFIJOS = True

if CORRER_EXPERIMENTO_PREFIJOS:
    col_pref = get_collection(COLLECTION_NAME + "_prefijos")
    indexar(col_pref, chunks_por_doc, prefijo_doc="search_document: ")
    ranks_pref = [
        rank_evidencia(recuperar(col_pref, p["pregunta"], prefijo_query="search_query: "), p["evidencia"])
        for p in con_evidencia
    ]
    comparacion = pd.DataFrame(
        [metricas_recuperacion(ranks_base), metricas_recuperacion(ranks_pref)],
        index=["sin prefijos (base)", "con prefijos"],
    )
    comparacion.to_csv(RESULTADOS_DIR / "experimento_prefijos.csv")
    display(comparacion)
    display(pd.DataFrame({"id": [p["id"] for p in con_evidencia],
                          "rank base": ranks_base, "rank con prefijos": ranks_pref}))

  + T01. Lógica Proposicional.pdf: 36 chunks (3.3s)
  + T02. Conjuntos.pdf: 29 chunks (2.4s)
  + T03. Números reales.pdf: 51 chunks (4.2s)
  + T04. Demostraciones.pdf: 20 chunks (1.6s)
  + T05. Números Complejos.pdf: 19 chunks (1.5s)
  + T06. Expresiones Algebraicas Polinómicas.pdf: 33 chunks (2.7s)
  + T07. Exp Alg no polinómicas. Ecuaciones e Inecuaciones.pdf: 26 chunks (2.2s)
  + T08. Plano cartesiano.pdf: 47 chunks (3.9s)
  + T09. Funciones.pdf: 63 chunks (5.1s)
  + T11. Trigonometría.pdf: 52 chunks (4.3s)
  + T12. Matrices (1C 2026).pdf: 5 chunks (0.4s)
  + T13. Sistemas (1C 2026).pdf: 13 chunks (1.1s)

Chunks nuevos: 394. Total en la colección: 394


,hit@1,hit@3,hit@5,hit@10,MRR
sin prefijos (base),0.1,0.5,0.7,0.9,0.35
con prefijos,0.2,0.6,0.6,0.9,0.43


,id,rank base,rank con prefijos
0,P1,9.0,9.0
1,P2,3.0,2.0
2,P3,5.0,6.0
3,P4,1.0,2.0
4,P5,2.0,2.0
5,P6,5.0,3.0
6,P7,2.0,1.0
7,P8,NaN,NaN
8,P9,6.0,6.0
9,P10,2.0,1.0


## 9. Conclusiones

> Completar con los números obtenidos en las secciones 7 y 8.

**Recuperación.** hit@10 = __ y MRR = __ sobre las 10 preguntas con evidencia. Preguntas en las que el retriever falló: __. Causa probable: __.

**CON vs. SIN RAG.** Nota media SIN RAG = __; CON RAG = __. La mejora se concentra en las preguntas *específicas*, porque dependen de recursos propios del curso (la analogía de la bolsa, el orden V-V-F-F, la construcción de la tabla trigonométrica, la noción de "condición inicial") que el modelo no puede conocer. En las *generales* la diferencia es __.

**Abstención.** En P11 (fuera de corpus), el modelo SIN RAG __; CON RAG __.

**Prefijos de `nomic-embed-text`.** __.

**Limitaciones observadas** (detalladas en el README):

- El chunking por caracteres corta palabras y fórmulas a la mitad.
- Las fórmulas LaTeX se extraen como texto plano desordenado (fracciones, raíces, matrices).
- La página citada es la de **inicio** del chunk, así que puede diferir de la página exacta de la respuesta.
- La corrección de las respuestas es manual y con pocas preguntas.